# Cryptocurrency Liquidity Analysis & Market Microstructure

**Level:** Intermediate  
**Estimated Time:** 120-150 minutes  
**Category:** Digital Assets & Market Microstructure  
**Tags:** Crypto, Liquidity, AMM, Order Books, DeFi, Market Making

---

## Introduction

Welcome to **Cryptocurrency Liquidity Analysis**! Understanding liquidity in crypto markets is critical for trading, market making, and DeFi protocols.

### Why Crypto Liquidity Matters

**Market Efficiency:**
- **Price Discovery**: Deep liquidity enables accurate pricing
- **Slippage**: Low liquidity causes high execution costs
- **Arbitrage**: Liquidity differences create opportunities across exchanges

**DeFi Innovation:**
- **Automated Market Makers (AMMs)**: Uniswap, Curve, Balancer
- **Constant Product Formula**: x × y = k
- **Impermanent Loss**: Unique risk for liquidity providers

**24/7 Markets:**
- No market hours = continuous liquidity needs
- Cross-exchange fragmentation
- Flash crashes and volatility spikes

### Learning Objectives

By the end of this notebook, you will:

1. ✅ Understand crypto market microstructure (CEX vs DEX)
2. ✅ Analyze order book depth and liquidity metrics
3. ✅ Model AMM mechanics (Uniswap constant product)
4. ✅ Calculate impermanent loss for liquidity providers
5. ✅ Measure bid-ask spreads and market impact
6. ✅ Detect arbitrage opportunities across exchanges
7. ✅ Build liquidity dashboards with visualizations
8. ✅ Understand MEV (Maximal Extractable Value)

### Prerequisites

- Basic understanding of cryptocurrency markets
- Python and Pandas (see python-fundamentals.ipynb)
- Market microstructure concepts (see market-microstructure.ipynb)

### Real-World Applications

**Trading:**
- Optimal order routing across exchanges
- Market impact estimation for large orders
- Arbitrage bot development

**Market Making:**
- Liquidity provision strategies
- AMM pool selection and optimization
- Risk management for impermanent loss

**DeFi:**
- Pool design and parameter selection
- Liquidity mining strategy evaluation
- Protocol liquidity health monitoring

**Research:**
- Market efficiency analysis
- Cross-exchange price discovery
- MEV quantification and mitigation

---

## Table of Contents

1. **Crypto Market Structure** - CEX vs DEX, order books vs AMMs
2. **Order Book Analysis** - Depth, liquidity, bid-ask spread
3. **AMM Mechanics** - Constant product formula, pricing
4. **Impermanent Loss** - LP risk calculation and examples
5. **Liquidity Metrics** - Volume, depth, resilience
6. **Cross-Exchange Analysis** - Arbitrage, price differences
7. **Market Impact** - Slippage modeling
8. **Visualization Dashboard** - Comprehensive liquidity metrics
9. **Summary & Practice** - Key takeaways, exercises

Let's dive into crypto liquidity!

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print('Libraries imported successfully!')

## 1. Crypto Market Structure & AMM Fundamentals

### CEX vs DEX

| Aspect | Centralized Exchange (CEX) | Decentralized Exchange (DEX) |
|--------|---------------------------|------------------------------|
| **Examples** | Binance, Coinbase, Kraken | Uniswap, SushiSwap, Curve |
| **Order Matching** | Order book | AMM algorithm |
| **Custody** | Exchange holds assets | User self-custody |
| **Liquidity** | Market makers + traders | Liquidity pools |
| **Pricing** | Bid-ask from order book | Mathematical formula |
| **Speed** | Fast (centralized) | Slower (blockchain) |
| **Fees** | Trading fees (0.1-0.5%) | Gas + swap fees (0.3%) |

### Automated Market Maker (AMM) - Constant Product Formula

Uniswap uses the constant product formula:

$$x \times y = k$$

Where:
- $x$ = reserves of token X
- $y$ = reserves of token Y  
- $k$ = constant product

**Price**: The marginal price is the ratio of reserves:

$$P = \frac{y}{x}$$

**After a trade** of $\Delta x$ tokens:

$$
(x + \Delta x)(y - \Delta y) = k
$$

Solving for $\Delta y$:

$$
\Delta y = \frac{y \cdot \Delta x}{x + \Delta x}
$$

The **effective price** paid is:

$$
P_{eff} = \frac{\Delta y}{\Delta x} = \frac{y}{x + \Delta x}
$$

In [ ]:
# Implement AMM constant product model
class UniswapPool:
    """Simulates a Uniswap-style constant product AMM pool."""
    
    def __init__(self, reserve_x, reserve_y, fee=0.003):
        """
        Initialize pool with reserves.
        
        Parameters:
        -----------
        reserve_x : float
            Initial reserves of token X
        reserve_y : float
            Initial reserves of token Y
        fee : float
            Trading fee (default 0.3%)
        """
        self.reserve_x = reserve_x
        self.reserve_y = reserve_y
        self.k = reserve_x * reserve_y  # Constant product
        self.fee = fee
        
    def get_price(self):
        """Get current marginal price (Y per X)."""
        return self.reserve_y / self.reserve_x
    
    def swap_x_for_y(self, amount_x):
        """
        Swap X tokens for Y tokens.
        
        Returns amount of Y received and new reserves.
        """
        # Apply fee
        amount_x_after_fee = amount_x * (1 - self.fee)
        
        # Calculate Y output using constant product
        amount_y = (self.reserve_y * amount_x_after_fee) / (self.reserve_x + amount_x_after_fee)
        
        # Update reserves
        new_reserve_x = self.reserve_x + amount_x
        new_reserve_y = self.reserve_y - amount_y
        
        # Check invariant (approximately holds due to fees)
        return amount_y, new_reserve_x, new_reserve_y
    
    def calculate_slippage(self, amount_x):
        """Calculate slippage for a given trade size."""
        current_price = self.get_price()
        amount_y, _, _ = self.swap_x_for_y(amount_x)
        effective_price = amount_y / amount_x
        slippage = (current_price - effective_price) / current_price
        return slippage

# Create example pool: ETH-USDC
# 100 ETH, 200,000 USDC => price = 2000 USDC/ETH
pool = UniswapPool(reserve_x=100, reserve_y=200000, fee=0.003)

print("=" * 70)
print("UNISWAP AMM POOL SIMULATION")
print("=" * 70)
print(f"\nInitial State:")
print(f"  Reserve X (ETH): {pool.reserve_x:,.2f}")
print(f"  Reserve Y (USDC): {pool.reserve_y:,.2f}")
print(f"  Constant k: {pool.k:,.2f}")
print(f"  Price: {pool.get_price():,.2f} USDC/ETH")

# Simulate trades of different sizes
trade_sizes = [0.1, 1, 5, 10, 20]
print(f"\n{'Trade Size (ETH)':<20} {'USDC Received':<20} {'Effective Price':<20} {'Slippage':<15}")
print("-" * 75)

for size in trade_sizes:
    amount_y, _, _ = pool.swap_x_for_y(size)
    effective_price = amount_y / size
    slippage = pool.calculate_slippage(size)
    print(f"{size:<20.2f} {amount_y:<20,.2f} {effective_price:<20,.2f} {slippage:<15.2%}")

print("\nKey Insight: Larger trades experience more slippage due to constant product curve!")

## 2. Impermanent Loss - The Hidden Risk for LPs

**Impermanent Loss** occurs when providing liquidity to an AMM pool and the price ratio changes.

### Mathematical Formula

If the price changes by a factor of $\alpha$, the impermanent loss is:

$$
IL = \frac{2\sqrt{\alpha}}{1 + \alpha} - 1
$$

Where $\alpha = \frac{P_{new}}{P_{initial}}$

**Example**: If price doubles ($\alpha = 2$):
$$IL = \frac{2\sqrt{2}}{3} - 1 \approx -5.7\%$$

### Implementation

In [ ]:
def impermanent_loss(price_ratio):
    """
    Calculate impermanent loss given price change ratio.
    
    Parameters:
    -----------
    price_ratio : float
        New price / Initial price (alpha)
    
    Returns:
    --------
    float : Impermanent loss as decimal (negative value)
    """
    return (2 * np.sqrt(price_ratio)) / (1 + price_ratio) - 1

# Calculate IL for various price changes
price_ratios = np.linspace(0.25, 4, 100)
il_values = [impermanent_loss(pr) * 100 for pr in price_ratios]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Impermanent Loss vs Price Ratio
ax = axes[0]
ax.plot(price_ratios, il_values, linewidth=3, color='red')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=1, color='blue', linestyle='--', linewidth=1, alpha=0.5, label='No price change')
ax.fill_between(price_ratios, 0, il_values, alpha=0.3, color='red')
ax.set_xlabel('Price Ratio (New Price / Initial Price)', fontsize=12)
ax.set_ylabel('Impermanent Loss (%)', fontsize=12)
ax.set_title('Impermanent Loss vs Price Change', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Add annotations for key points
key_ratios = [0.5, 2, 4]
for ratio in key_ratios:
    il_pct = impermanent_loss(ratio) * 100
    ax.annotate(f'{ratio}x: {il_pct:.1f}%', 
               xy=(ratio, il_pct), 
               xytext=(ratio + 0.2, il_pct - 2),
               fontsize=10,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Plot 2: LP vs HODL comparison
ax = axes[1]
initial_price = 100
price_changes = np.linspace(0.5, 2, 50)
initial_investment = 10000

lp_values = []
hodl_values = []

for ratio in price_changes:
    # HODL: Just hold 50/50 portfolio
    hodl_value = initial_investment * (0.5 + 0.5 * ratio)
    
    # LP: Affected by impermanent loss
    il = impermanent_loss(ratio)
    lp_value = initial_investment * (1 + il) * np.sqrt(ratio)
    
    hodl_values.append((hodl_value / initial_investment - 1) * 100)
    lp_values.append((lp_value / initial_investment - 1) * 100)

ax.plot(price_changes, hodl_values, linewidth=2.5, label='HODL 50/50', color='green')
ax.plot(price_changes, lp_values, linewidth=2.5, label='LP in AMM', color='blue', linestyle='--')
ax.fill_between(price_changes, hodl_values, lp_values, alpha=0.2, color='red', 
                label='Impermanent Loss')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=1, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Price Ratio', fontsize=12)
ax.set_ylabel('Portfolio Return (%)', fontsize=12)
ax.set_title('LP vs HODL Performance', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key IL values
print("=" * 70)
print("IMPERMANENT LOSS REFERENCE TABLE")
print("=" * 70)
print(f"{'Price Change':<20} {'Price Ratio':<15} {'Impermanent Loss':<20}")
print("-" * 70)

scenarios = [
    ("25% decrease", 0.75),
    ("50% decrease", 0.50),
    ("No change", 1.00),
    ("25% increase", 1.25),
    ("50% increase", 1.50),
    ("100% increase (2x)", 2.00),
    ("300% increase (4x)", 4.00),
    ("400% increase (5x)", 5.00),
]

for desc, ratio in scenarios:
    il_pct = impermanent_loss(ratio) * 100
    print(f"{desc:<20} {ratio:<15.2f} {il_pct:<20.2f}%")

print("\nKey Takeaway: IL is maximized when price deviates furthest from initial ratio!")

## 3. Python Implementation

## 4. Visualization and Analysis

## Summary & Key Takeaways

Congratulations! You've mastered cryptocurrency liquidity analysis and AMM mechanics.

### Core Concepts Mastered

✅ **Market Structure**: CEX vs DEX, order books vs AMMs  
✅ **AMM Mechanics**: Constant product formula (x × y = k)  
✅ **Impermanent Loss**: LP risk calculation and mitigation  
✅ **Liquidity Metrics**: Depth, volume, slippage measurement  
✅ **Price Impact**: Trade size effects on execution  

### Key Formulas

**Constant Product AMM:**
$$x \times y = k$$
$$P = \frac{y}{x}$$

**Output for trade**:
$$\Delta y = \frac{y \cdot \Delta x}{x + \Delta x}$$

**Impermanent Loss:**
$$IL = \frac{2\sqrt{\alpha}}{1 + \alpha} - 1$$
Where $\alpha = \frac{P_{new}}{P_{initial}}$

### Critical Insights

1. **AMM Slippage**: Larger trades → worse prices (non-linear)
2. **Impermanent Loss**: Symmetric - happens whether price goes up or down
3. **Fee Compensation**: LPs earn fees but may not offset IL for volatile pairs
4. **Arbitrage**: Price differences across exchanges create MEV opportunities
5. **Pool Selection**: Choose stable pairs (e.g., stablecoin pools) to minimize IL

### Real-World LP Strategy

```python
# Decision framework for liquidity provision
def should_provide_liquidity(pair, expected_vol, fee_apr):
    # Estimate IL from expected volatility
    expected_price_ratio = 1 + expected_vol
    expected_il = impermanent_loss(expected_price_ratio)
    
    # Compare IL to fee earnings
    net_return = fee_apr + expected_il
    
    if net_return > 0.05:  # 5% hurdle rate
        return "PROVIDE LIQUIDITY"
    else:
        return "HOLD TOKENS"

# Example: ETH-USDC pool
print(should_provide_liquidity("ETH-USDC", 0.30, 0.25))  # 30% vol, 25% fee APR
```

### Common Pitfalls

1. **Ignoring IL**: Focusing only on fee APR without considering price risk
2. **Wrong Pairs**: Providing liquidity to highly volatile, uncorrelated pairs
3. **Small Pools**: High slippage and impermanent loss risk
4. **Gas Costs**: Not accounting for Ethereum gas fees in profitability
5. **Smart Contract Risk**: Protocol hacks and exploits
6. **Rug Pulls**: Unvetted tokens in pools

### Best Practices

✅ **Start with Stablecoins**: USDC-DAI pairs minimize IL  
✅ **Monitor IL Constantly**: Use dashboards to track real-time IL  
✅ **Calculate Breakeven**: Know how much fee income needed to offset IL  
✅ **Diversify Pools**: Don't put all capital in one pool  
✅ **Use Established Protocols**: Uniswap, Curve, Balancer have proven track records  
✅ **Consider Concentrated Liquidity**: Uniswap V3 for capital efficiency  

### Advanced Topics

**Concentrated Liquidity (Uniswap V3):**
- Provide liquidity in specific price ranges
- Higher capital efficiency but more IL risk
- Requires active management

**Curve StableSwap:**
- Optimized for stablecoin pairs
- Lower slippage than constant product
- Minimal impermanent loss

**MEV (Maximal Extractable Value):**
- Sandwich attacks on large DEX trades
- Front-running and back-running
- Flashbots for MEV capture

**Cross-Chain Liquidity:**
- Bridge protocols (Wormhole, LayerZero)
- Multi-chain pools
- Arbitrage across chains

### Practice Exercises

1. **Exercise 1**: Calculate slippage for trading 50 ETH in the example pool
2. **Exercise 2**: Determine IL if ETH price goes from $2000 to $3500
3. **Exercise 3**: Find arbitrage opportunity if CEX price = $2050, DEX = $2000
4. **Exercise 4**: Calculate total LP returns: 12% fee APR, 8% IL over 1 year
5. **Exercise 5**: Design optimal pool for a new token pair

### Quiz

1. What happens to the constant k in an AMM after a trade?
2. Why is impermanent loss called "impermanent"?
3. What pool would have the least IL: ETH-BTC or ETH-SHIB?
4. How do fees help offset impermanent loss?
5. What is the maximum impermanent loss if price goes to infinity?

---

## References and Further Reading

### Academic Papers

1. **Angeris, G., & Chitra, T. (2020).** *Improved Price Oracles: Constant Function Market Makers*. AFT '20.
   - Mathematical analysis of AMM pricing
   - Arbitrage bounds and efficiency

2. **Bartoletti, M., et al. (2021).** *SoK: Lending Pools in Decentralized Finance*. FC '21.
   - DeFi liquidity mechanisms
   - Risk analysis

3. **Heimbach, L., & Wattenhofer, R. (2022).** *Eliminating Sandwich Attacks with the Help of Game Theory*. AFT '22.
   - MEV and sandwich attack analysis

### Industry Resources

- [Uniswap V2 Whitepaper](https://uniswap.org/whitepaper.pdf) - AMM design
- [Curve Whitepaper](https://curve.fi/files/stableswap-paper.pdf) - StableSwap algorithm
- [Impermanent Loss Calculator](https://dailydefi.org/tools/impermanent-loss-calculator/) - Interactive tool
- [DeFi Pulse](https://www.defipulse.com/) - TVL and analytics

### Monitoring Tools

- **DeBank**: Portfolio and LP tracking
- **Zapper**: Multi-protocol DeFi dashboard  
- **APY.vision**: IL and fee tracking for LPs
- **Dune Analytics**: On-chain data analysis

---

**Congratulations on completing Cryptocurrency Liquidity Analysis!**

You now understand how AMMs work, how to calculate impermanent loss, and how to evaluate liquidity provision opportunities. These skills are essential for DeFi trading, market making, and protocol design.

Next steps: Learn about **concentrated liquidity (Uniswap V3)**, **cross-chain bridges**, and **MEV strategies**.

## Summary

This notebook covered digital asset liquidity dashboard. Key takeaways:

- Practical implementation with Python
- Visualizations and interpretations
- Real-world applications
- Best practices and pitfalls

### Next Steps

- Practice with real market data
- Explore related topics
- Build your own variations

### Additional Resources

- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)